# Multi-Query RAG

<img src="../images/Multi-Query RAG.png" width=“500”>

In [66]:
from langchain_core.globals import set_debug
set_debug(False)

## Initialize Azure Chat OpenAI

In [53]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

load_dotenv(override=True, verbose=True)

chat_llm = AzureChatOpenAI(
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_deployment=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')
)

## Initialize Azure Embedding Open AI

In [54]:
emd_llm = AzureOpenAIEmbeddings(
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_deployment=os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
)

## Load the Document and upload it to vector store

In [55]:
from langchain_community.document_loaders import PyMuPDFLoader

file = '../Data/HTTP-Status-Codes.pdf'
pdf_docs = PyMuPDFLoader(file_path=file).load()


In [56]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 50
)
chunks = splitter.split_documents(documents=pdf_docs)

In [57]:
from langchain_community.vectorstores import FAISS

faiss = FAISS.from_documents(documents=chunks, embedding=emd_llm)


## Multi Query generation from User's query

### Multi query Prompt

In [67]:
MULTI_QUERY_PROMPT = """"
You are a helpful assistant that generates alternative phrasings of a user's question to improve document retrieval from a vector database.
Produce {num_queries} different versions that capture distinct phrasings and perspectives of the question.
Important: Generate one query per line, output ONE query per line, with no numbering, bullets, or extra text.
"""

In [68]:
from langchain_core.prompts import ChatPromptTemplate

multi_query_prompt_template = ChatPromptTemplate.from_messages(
    [
    ("system", MULTI_QUERY_PROMPT),
    ("user", "Original questions: {question}"),
    ]
)

In [69]:
num_queries = 3
question = "What is error code 404?"
prompt = multi_query_prompt_template.invoke({"num_queries": num_queries, "question": question})

### Invoke Chat LLM to generate the queries

In [70]:
response = chat_llm.invoke(prompt)
print(response.content)

What does the HTTP 404 error code mean?
Why does a website return a 404 Not Found response and how can it be fixed?
Explain the meaning and common causes of a 404 error in web browsing


In [71]:
multiple_queries = [q for q in response.content.split("\n")]
print(f"total queries generated: {len(multiple_queries)}")
for q in multiple_queries:
    print(q)

total queries generated: 3
What does the HTTP 404 error code mean?
Why does a website return a 404 Not Found response and how can it be fixed?
Explain the meaning and common causes of a 404 error in web browsing


## Call to Vector store to get the relevant chunks

In [72]:
fetched_response = []
for q in multiple_queries:
    similar_docs = faiss.similarity_search(
        query=q, 
        k = 3)
    print(f"query: {q}", end="\n")
    for sd in similar_docs:
        print(sd.page_content, end="\n\n")
        fetched_response.append(sd.page_content)
    print("\n------\n")

query: What does the HTTP 404 error code mean?
permanent.
4XX codes flag client errors. You typed a wrong URL. You don’t have
permission. The page doesn’t exist. These are on you or your users.

200 OK when it works or 404 Not Found when the resource doesn’t exist.
How to Check HTTP Status Code Using Tools
You need to see status codes when debugging. Several tools make this easy.

404 Not Found Error Impact
410 Gone vs 404 Error Codes
500 Internal Server Error Solutions
503 Service Unavailable Fix
Common HTTP Status Code FAQ
Search...
Tutorials
Comparisons
News
Keep in touch


------

query: Why does a website return a 404 Not Found response and how can it be fixed?
They bounce. Your site looks broken.
Fix 404 errors immediately. If you deleted a page intentionally, redirect it to
relevant content with a 301. If the page should exist, figure out why it doesn’t

200 OK when it works or 404 Not Found when the resource doesn’t exist.
How to Check HTTP Status Code Using Tools
You need to s

In [73]:
PROMPT_TEMPLATE = """"
You are a helpful assistance. You answer user's query {question} from the provided context {context}. If context isn't sufficient to provide the answer, inform the user that Context isn't sufficient to provide the answer of your query.
"""
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", PROMPT_TEMPLATE),
        ("user",  "user query: {question}"),
    ]
)


In [74]:
prompt =  prompt_template.invoke({'question': question, 'context': fetched_response})
response = chat_llm.invoke(prompt)
print(response)

print(response.content)

content='A 404 (Not Found) is an HTTP status code meaning the server can’t find the requested resource. It’s in the 4XX family (client errors), so common causes are a mistyped URL, a deleted or moved page, or a permissions/ownership issue on the client side.\n\nKey points from the context:\n- 404 = resource doesn’t exist (right now); could be a typo or temporary.\n- A 410 (Gone) is a stronger, permanent signal that a page was intentionally removed.\n- 404s hurt SEO (pages can be dropped from search indexes, backlinks lose value) and frustrate users.\n- Fixes: if a page was removed on purpose, redirect it to relevant content with a 301; if it should exist, restore it or correct the URL/permissions; use tools to check HTTP status codes when debugging.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 761, 'prompt_tokens': 453, 'total_tokens': 1214, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_token